# Human Development Index (HDI) Prediction
**A Comprehensive Measure of Well-Being**

This Jupyter Notebook walks through the step-by-step implementation of the machine learning pipeline to predict a country's Human Development Index (HDI) using **Linear Regression** based on standard socio-economic metrics: life expectancy, average schooling years, gross national income per capita, and internet usage.

## Epic 2: Importing Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder
import pickle
import os
import sys

# Set plots style
sns.set_style("whitegrid")
print("All required libraries imported successfully.")

## Epic 3: Dataset Download and Understanding

In [ ]:
# Import preprocessing scripts from local modules
sys.path.append(os.path.abspath('.'))
import preprocessing

# Ensure dataset is downloaded and load it
filepath = preprocessing.download_dataset_if_missing(dataset_dir="../Dataset", filename="HDI.csv")
df = preprocessing.load_data(filepath)

# Display first 5 rows
df.head()

In [ ]:
# Inspect dataset structure
df.info()

In [ ]:
# Summary statistics
df.describe()

## Epic 4: Data Preprocessing and Label Encoding

In [ ]:
# Select features and target
X, y = preprocessing.get_features_and_target(df)

print("Features selected:", list(X.columns))
print("Target name:", y.name)
print(f"Missing values before handling:\n{X.isnull().sum()}")

In [ ]:
# Impute missing values with column mean
X_clean = preprocessing.handle_missing_values(X)
print(f"Missing values after imputation:\n{X_clean.isnull().sum()}")

In [ ]:
# Label encode the categorical variable 'Country'
X_encoded, le = preprocessing.encode_categorical_features(X_clean)
X_encoded.head()

## Data Visualization (EDA)

In [ ]:
# Generate visualizations
import visualization

# Define training output directory for local visual inspection
os.makedirs("static/images", exist_ok=True)

# Heatmap of features
visualization.plot_correlation_heatmap(X_encoded, y, ["static/images"])

# Distribution plot of target variable
visualization.plot_distribution_plots(y, ["static/images"])

# Strip plots
visualization.plot_strip_plots(X_clean, y, ["static/images"])

# Use Inline matplotlib for notebooks
%matplotlib inline
plt.figure(figsize=(8,6))
sns.heatmap(X_encoded.drop(columns=["Country"]).join(y).corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap (Interactive View)")
plt.show()

## Epic 5: Dividing the Dataset into Train and Test Data

In [ ]:
# Split dataset (90% train, 10% test, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.1, random_state=42)
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}, y_test shape:  {y_test.shape}")

## Epic 6: Fitting the Model

In [ ]:
import model

# Train linear regression
reg = model.train_linear_regression(X_train, y_train)

In [ ]:
# Evaluate model on test data
metrics, y_pred = model.evaluate_model(reg, X_test, y_test)

In [ ]:
# Plot actual vs predicted values
visualization.plot_actual_vs_predicted(y_test.values, y_pred, ["static/images"])

# Interactive notebook scatter plot
plt.figure(figsize=(7,6))
plt.scatter(y_test, y_pred, color="green", alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
plt.title("Actual vs Predicted HDI")
plt.xlabel("Actual HDI")
plt.ylabel("Predicted HDI")
plt.show()

## Epic 7: Saving the Model

In [ ]:
# Package and serialize the model to Flask directory
os.makedirs("../Flask", exist_ok=True)
model.save_serialized_model(reg, le, list(le.classes_), "../Flask/HDI.pkl")